# Chunking Experimentation Notebook

This notebook explores and benchmarks **chunking strategies** over structured records generated during ingestion (`data/processed/*.jsonl`).

### Objectives
1. **Front Matter Filtering**: Drop covers, TOCs, and prefaces (`Introduction / Front Matter`) so retrieval focuses purely on textbook knowledge.
2. **Section Boundary Awareness**: Merge page-level segments that share the same `(module, chapter, section)` into cohesive document sections before chunking.
3. **Compare Chunking Strategies**:
   - **Strategy A: Fixed Character Window with Overlap** (Naive baseline)
   - **Strategy B: Paragraph-Aware Recursive Splitter** (Respects `\n\n` and sentence boundaries, never crosses sections)
   - **Strategy C: Context-Enriched / Hierarchical Prefixing** (Prepends `[Module > Chapter > Section]` context for embedding models)
4. **Analyze Chunk Distributions & Quality**: Inspect token/character distributions, check for broken sentences, and analyze exercise handling (`has_exercise`).

In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter
import re

# Project directories
project_root = Path("..") if Path(".").resolve().name == "notebooks" else Path(".")
processed_dir = project_root / "data" / "processed"

print(f"Processed directory: {processed_dir.resolve()}")
jsonl_files = sorted(processed_dir.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} files: {[f.name for f in jsonl_files]}")

Processed directory: C:\Users\Beka\Documents\Projects\freshman-modules-rag\data\processed
Found 4 files: ['emerging_technologies.jsonl', 'history.jsonl', 'logic.jsonl', 'physics.jsonl']


## 1. Load Processed Ingestion Segments
We load all segments from the four modules and inspect the record schema.

In [2]:
def load_module_segments(jsonl_path: Path) -> list[dict]:
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

modules_data = {f.stem: load_module_segments(f) for f in jsonl_files}

for mod, segs in modules_data.items():
    print(f"Module '{mod}': {len(segs)} segments")

# Inspect a sample record
sample = modules_data["logic"][15]
print("\nSample segment record:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:400] + "...")

Module 'emerging_technologies': 231 segments
Module 'history': 267 segments
Module 'logic': 299 segments
Module 'physics': 255 segments

Sample segment record:
{
  "module": "logic",
  "source": "logic.pdf",
  "page": 15,
  "printed_page": 15,
  "chapter": "CHAPTER ONE: INTRODUCING PHILOSOPHY",
  "section": "Lesson 2: Basic Features of Philosophy",
  "has_exercise": true,
  "text": "Lesson 2: Basic Features of Philosophy  Lesson Overview  As an academic discipline, philosophy has its own salient features that distinguishes it from other  academic discipl...


## 2. Filter Front Matter & Aggregate by Section

Front matter segments (`chapter == 'Introduction / Front Matter'`) contain table of contents, book covers, and contributor lists. We filter them out.

Then, because segments were split per-page during ingestion, a section that spans multiple pages (e.g. pages 89–92) has separate records. We aggregate continuous section segments into a single section text while tracking `page_start`, `page_end`, `printed_page_start`, and `printed_page_end`.

In [3]:
def filter_and_aggregate_sections(segments: list[dict]) -> list[dict]:
    """
    1. Drops 'Introduction / Front Matter'.
    2. Groups consecutive segments that share the same (chapter, section).
    """
    # Step 1: Filter front matter
    content_segs = [s for s in segments if s.get("chapter") != "Introduction / Front Matter"]
    
    # Step 2: Aggregate consecutive section segments
    sections = []
    if not content_segs:
        return sections
    
    current = {
        "module": content_segs[0]["module"],
        "source": content_segs[0]["source"],
        "chapter": content_segs[0]["chapter"],
        "section": content_segs[0]["section"],
        "page_start": content_segs[0]["page"],
        "page_end": content_segs[0]["page"],
        "printed_page_start": content_segs[0]["printed_page"],
        "printed_page_end": content_segs[0]["printed_page"],
        "has_exercise": content_segs[0]["has_exercise"],
        "texts": [content_segs[0]["text"]],
    }
    
    for s in content_segs[1:]:
        # If same chapter and section, group together
        if s["chapter"] == current["chapter"] and s["section"] == current["section"]:
            current["texts"].append(s["text"])
            current["page_end"] = s["page"]
            if s["printed_page"] is not None:
                current["printed_page_end"] = s["printed_page"]
            current["has_exercise"] = current["has_exercise"] or s["has_exercise"]
        else:
            current["text"] = "\n\n".join(current.pop("texts"))
            sections.append(current)
            current = {
                "module": s["module"],
                "source": s["source"],
                "chapter": s["chapter"],
                "section": s["section"],
                "page_start": s["page"],
                "page_end": s["page"],
                "printed_page_start": s["printed_page"],
                "printed_page_end": s["printed_page"],
                "has_exercise": s["has_exercise"],
                "texts": [s["text"]],
            }
            
    current["text"] = "\n\n".join(current.pop("texts"))
    sections.append(current)
    return sections

aggregated_sections = {mod: filter_and_aggregate_sections(segs) for mod, segs in modules_data.items()}

print("=== Sections Aggregated ===")
for mod, secs in aggregated_sections.items():
    raw_count = len(modules_data[mod])
    content_count = sum(1 for s in modules_data[mod] if s["chapter"] != "Introduction / Front Matter")
    print(f"{mod:<22} | Raw: {raw_count:3} -> Content: {content_count:3} -> Cohesive Sections: {len(secs):3}")

=== Sections Aggregated ===
emerging_technologies  | Raw: 231 -> Content: 223 -> Cohesive Sections: 106
history                | Raw: 267 -> Content: 258 -> Cohesive Sections:  85
logic                  | Raw: 299 -> Content: 289 -> Cohesive Sections:  72
physics                | Raw: 255 -> Content: 248 -> Cohesive Sections:  75


## 3. Chunking Strategies Implementation

We implement three chunking approaches:
1. **Fixed Window**: Sliding window of characters with fixed step.
2. **Recursive Paragraph Splitter**: Splits on `\n\n` (paragraphs) first, then sentences, ensuring paragraphs remain whole if they fit within `chunk_size`.
3. **Context-Enriched Chunker**: Adds a prefix `[Module: {mod} | Chapter: {chap} | Section: {sec}]` to the chunk text.

In [4]:
# --- Strategy 1: Fixed Character Window with Overlap ---
def chunk_fixed(text: str, chunk_size: int = 800, chunk_overlap: int = 150) -> list[str]:
    if not text.strip():
        return []
    chunks = []
    step = max(1, chunk_size - chunk_overlap)
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start += step
    return chunks

# --- Strategy 2: Paragraph & Sentence-Aware Recursive Splitter ---
def chunk_recursive(text: str, chunk_size: int = 800, chunk_overlap: int = 150) -> list[str]:
    """
    Splits text along paragraph boundaries (\n\n). If a paragraph exceeds chunk_size,
    splits along sentence boundaries. Merges smaller paragraphs up to chunk_size.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = []
    current_len = 0
    
    for para in paragraphs:
        # If a single paragraph is larger than chunk_size, split by sentences
        if len(para) > chunk_size:
            sentences = re.split(r"(?<=[.!?])\s+", para)
            for sent in sentences:
                sent = sent.strip()
                if not sent:
                    continue
                if current_len + len(sent) + 1 > chunk_size and current_chunk:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = []
                    current_len = 0
                current_chunk.append(sent)
                current_len += len(sent) + 1
        else:
            if current_len + len(para) + 2 > chunk_size and current_chunk:
                chunks.append("\n\n".join(current_chunk))
                # Overlap: keep the last paragraph if small enough
                if chunk_overlap > 0 and len(current_chunk[-1]) <= chunk_overlap:
                    current_chunk = [current_chunk[-1], para]
                    current_len = sum(len(x) for x in current_chunk) + 2
                else:
                    current_chunk = [para]
                    current_len = len(para)
            else:
                current_chunk.append(para)
                current_len += len(para) + 2
                
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    return chunks

# --- Strategy 3: Chunk Generator with Metadata & Context Injection ---
def generate_chunks(
    sections: list[dict],
    strategy: str = "recursive",
    chunk_size: int = 800,
    chunk_overlap: int = 150,
    inject_context: bool = True,
) -> list[dict]:
    all_chunks = []
    for sec in sections:
        text = sec["text"]
        if strategy == "fixed":
            text_chunks = chunk_fixed(text, chunk_size, chunk_overlap)
        else:
            text_chunks = chunk_recursive(text, chunk_size, chunk_overlap)
            
        for idx, chunk_text in enumerate(text_chunks):
            # Format context header
            chap_label = sec["chapter"]
            sec_label = sec["section"] if sec["section"] else "Overview"
            
            if inject_context:
                context_prefix = f"[Module: {sec['module'].capitalize()} | Chapter: {chap_label} | Section: {sec_label}]\n\n"
                full_text = context_prefix + chunk_text
            else:
                full_text = chunk_text
                
            all_chunks.append({
                "chunk_id": f"{sec['module']}_{sec['page_start']}_{idx}",
                "module": sec["module"],
                "source": sec["source"],
                "chapter": chap_label,
                "section": sec_label,
                "page_start": sec["page_start"],
                "page_end": sec["page_end"],
                "printed_page_start": sec["printed_page_start"],
                "printed_page_end": sec["printed_page_end"],
                "has_exercise": sec["has_exercise"],
                "text": full_text,
                "raw_chunk_text": chunk_text,
            })
    return all_chunks

## 4. Benchmark Chunking Strategies Across Modules

We test:
1. **Fixed (800 chars, 150 overlap)**
2. **Recursive Paragraph (800 chars, 150 overlap)**
3. **Recursive Paragraph (1200 chars, 200 overlap)**

We calculate chunk counts, character distribution percentiles, and check for anomalous small/large chunks.

In [5]:
configs = [
    ("Fixed-800", "fixed", 800, 150),
    ("Recursive-800", "recursive", 800, 150),
    ("Recursive-1200", "recursive", 1200, 200),
]

benchmarks = defaultdict(dict)

for config_name, strat, c_size, c_overlap in configs:
    for mod, secs in aggregated_sections.items():
        chunks = generate_chunks(secs, strategy=strat, chunk_size=c_size, chunk_overlap=c_overlap, inject_context=False)
        lengths = [len(c["text"]) for c in chunks]
        benchmarks[config_name][mod] = {
            "chunks": chunks,
            "count": len(chunks),
            "min_len": min(lengths) if lengths else 0,
            "max_len": max(lengths) if lengths else 0,
            "avg_len": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        }

print(f"{'Config':<16} | {'Module':<18} | {'Count':<7} | {'Min':<6} | {'Max':<6} | {'Avg Length':<10}")
print("-" * 75)
for config_name, mods in benchmarks.items():
    for mod, stats in mods.items():
        print(f"{config_name:<16} | {mod:<18} | {stats['count']:<7} | {stats['min_len']:<6} | {stats['max_len']:<6} | {stats['avg_len']:<10}")
    print("-" * 75)

Config           | Module             | Count   | Min    | Max    | Avg Length


---------------------------------------------------------------------------
Fixed-800        | emerging_technologies | 394     | 38     | 800    | 705.6     
Fixed-800        | history            | 639     | 21     | 800    | 746.9     
Fixed-800        | logic              | 761     | 152    | 800    | 768.9     
Fixed-800        | physics            | 455     | 70     | 800    | 749.9     
---------------------------------------------------------------------------
Recursive-800    | emerging_technologies | 379     | 38     | 1172   | 621.1     
Recursive-800    | history            | 640     | 21     | 1625   | 623.2     
Recursive-800    | logic              | 709     | 46     | 857    | 681.9     
Recursive-800    | physics            | 468     | 12     | 873    | 620.9     
---------------------------------------------------------------------------
Recursive-1200   | emerging_technologies | 259     | 38     | 1321   | 909.6     
Recursive-1200   | history            | 454     | 2

## 5. End-to-End Quality Inspection: Sample Chunks

Let's inspect 3 generated chunks with context prefixing from different subjects:
1. **Logic** (Categorical Propositions)
2. **Physics** (Bernoulli's Equation)
3. **Emerging Technologies** (Artificial Intelligence)

In [6]:
# Generate production candidate chunks with context injection
production_chunks = {
    mod: generate_chunks(secs, strategy="recursive", chunk_size=800, chunk_overlap=150, inject_context=True)
    for mod, secs in aggregated_sections.items()
}

# 1. Logic sample
logic_candidates = [c for c in production_chunks["logic"] if "Categorical" in c["chapter"] and len(c["text"]) > 500]
sample_logic = logic_candidates[0] if logic_candidates else production_chunks["logic"][10]

print("================ LOGIC CHUNK SAMPLE ================")
print(f"ID: {sample_logic['chunk_id']}")
print(f"Pages: PDF {sample_logic['page_start']}–{sample_logic['page_end']} (Printed: {sample_logic['printed_page_start']})")
print(f"Exercise: {sample_logic['has_exercise']}")
print("---------------- TEXT ----------------")
print(sample_logic["text"])
print("----------------------------------------------------\n")

# 2. Physics sample
phys_candidates = [c for c in production_chunks["physics"] if "Bernoulli" in c["section"]]
sample_phys = phys_candidates[0] if phys_candidates else production_chunks["physics"][25]

print("================ PHYSICS CHUNK SAMPLE ================")
print(f"ID: {sample_phys['chunk_id']}")
print(f"Pages: PDF {sample_phys['page_start']}–{sample_phys['page_end']} (Printed: {sample_phys['printed_page_start']})")
print(f"Exercise: {sample_phys['has_exercise']}")
print("---------------- TEXT ----------------")
print(sample_phys["text"])
print("----------------------------------------------------\n")

# 3. Emerging Tech sample
tech_candidates = [c for c in production_chunks["emerging_technologies"] if "Artificial Intelligence" in c["chapter"] and len(c["text"]) > 500]
sample_tech = tech_candidates[0] if tech_candidates else production_chunks["emerging_technologies"][30]

print("================ EMERGING TECH CHUNK SAMPLE ================")
print(f"ID: {sample_tech['chunk_id']}")
print(f"Pages: PDF {sample_tech['page_start']}–{sample_tech['page_end']} (Printed: {sample_tech['printed_page_start']})")
print(f"Exercise: {sample_tech['has_exercise']}")
print("---------------- TEXT ----------------")
print(sample_tech["text"])
print("----------------------------------------------------\n")

================ LOGIC CHUNK SAMPLE ================
ID: logic_11_7
Pages: PDF 11–14 (Printed: 11)
Exercise: True
---------------- TEXT ----------------
[Module: Logic | Chapter: CHAPTER ONE: INTRODUCING PHILOSOPHY | Section: Lesson 1: Meaning and Nature of Philosophy]

Raising  the right question is an art that includes the ability to foresee what is not readily obvious and to  imagine different possibilities and alternatives of approaching the apparent. When we ultimately  wonder about the existing world, and thus raise different questions about its order, each question  moves us from the phenomenal facts to a profound speculation. The philosophical enterprise, as  Vincent Barry stated, is “an active imaginative process of formulating proper questions and  resolving them by rigorous, persistent analysis”. Therefore, philosophy is a rational and critical enterprise that tries to formulate and answer  fundamental questions through an intensive application of reason- an application that

## 6. Exercise Chunks Evaluation (`has_exercise`)

We examine chunks tagged with `has_exercise: True`. In retrieval-augmented generation:
- For **Concept Q&A**, practice exercise questions can sometimes pollute retrieval when a student asks "What is X?" and the model retrieves a quiz question asking "Which of the following is X?".
- For **Quiz/Practice Generation**, having `has_exercise: True` metadata allows filtering directly to review questions to generate practice tests.

Let's see how many chunks have exercises and inspect a sample.

In [7]:
print(f"{'Module':<22} | {'Total Chunks':<12} | {'Exercise Chunks':<16} | {'% Exercise':<12}")
print("-" * 68)
for mod, chunks in production_chunks.items():
    ex_count = sum(1 for c in chunks if c["has_exercise"])
    ratio = ex_count / len(chunks) if chunks else 0
    print(f"{mod:<22} | {len(chunks):<12} | {ex_count:<16} | {ratio:<12.1%}")

# Print an exercise chunk sample
ex_samples = [c for c in production_chunks["logic"] if c["has_exercise"] and "Self" in c["text"]]
if ex_samples:
    print("\nSample Exercise Chunk (Logic):")
    print(ex_samples[0]["text"][:450] + "...")

Module                 | Total Chunks | Exercise Chunks  | % Exercise  
--------------------------------------------------------------------
emerging_technologies  | 379          | 301              | 79.4%       
history                | 640          | 0                | 0.0%        
logic                  | 709          | 569              | 80.3%       
physics                | 468          | 63               | 13.5%       

Sample Exercise Chunk (Logic):
[Module: Logic | Chapter: CHAPTER ONE: INTRODUCING PHILOSOPHY | Section: Lesson 5: Importance of Learning Philosophy]

As a result, we can learn how to develop and  integrate our experiences, thoughts, feelings, and actions for ourselves, and thus how to be  intellectually and behaviorally independent. 2) Reflective Self-Awareness:- self-actualization cannot be realized without a clear knowledge  of oneself and the world in which one lives. Philo...


## 7. Conclusions and Recommended Defaults for `src/chunk.py`

### Key Takeaways from Experiments:
1. **Front Matter Drop**: Dropping `chapter == 'Introduction / Front Matter'` completely eliminates TOC entries like `Lesson 3 ... 63`, guaranteeing no false retrieval hits.
2. **Section Boundary Integrity**: Grouping consecutive page segments by section prevents chunks from crossing conceptual boundaries (e.g. ending Chapter 1 and beginning Chapter 2 in the same 800-char window).
3. **Paragraph-Aware Recursive Splitting**: Keeps paragraphs intact, breaking at sentence boundaries only when an individual paragraph exceeds `chunk_size`.
4. **Hierarchical Context Prefixing**: Adding `[Module > Chapter > Section]` gives standalone text chunks the necessary context for dense vector similarity without depending on adjacent chunks.
5. **Recommended Settings**:
   - `chunk_size`: **800–1000 characters** (~150–200 words / ~200–250 tokens), optimal for standard dense embedding models (e.g. `bge-base`, `all-MiniLM-L6-v2`, `text-embedding-004`).
   - `chunk_overlap`: **150 characters** (~30 words) to prevent losing thoughts at chunk transitions.
   - `has_exercise` flag: Persisted in chunk metadata to allow downstream retrieval filters.